# BioLRAF visualization workflow: GSE226365

This notebook reproduces the BioLRAF three-dimensional ternary visualization for the complete GSE226365 dataset. The same cells are displayed twice: colored by `celltype` and by `lineage_dominant`. The vertical position represents `Diff_score`.

The input is a cell-level score table containing `Os`, `Ch`, `Ad`, `MSC`, and `celltype`. The notebook writes two interactive HTML files, eight fixed-view PNG files, and a processed score table to `Differentiation_Models/GSE226365/`.

## 1. Import functions and locate the repository

In [ ]:
from pathlib import Path
import copy
import sys

import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
METHODS_DIR = CURRENT_DIR if CURRENT_DIR.name == "Methods" else CURRENT_DIR / "Methods"
if not (METHODS_DIR / "BioLRAF_visualization.py").exists():
    raise FileNotFoundError(
        "Cannot find Methods/BioLRAF_visualization.py. "
        "Start Jupyter from the BioLRAF repository root or its Methods directory."
    )

REPO_ROOT = METHODS_DIR.parent
sys.path.insert(0, str(METHODS_DIR))

from BioLRAF_visualization import (
    CAMERA_LIST,
    LINEAGE_COLORS,
    calculate_biolraf_metrics,
    configure_static_layout,
    plot_3d_ternary,
    read_score_table,
)

print(f"Repository root: {REPO_ROOT}")

## 2. Configuration

The preferred input location is `Differentiation_Models/GSE226365/GSE226365_mat.csv`. For compatibility, the notebook also checks `Example/GSE226365/input/`.

In [ ]:
DATASET_NAME = "GSE226365"
CENTER_DELTA = 0.10
S0 = 1.2
HTML_WIDTH = 820
HTML_HEIGHT = 700

OUTPUT_DIR = REPO_ROOT / "Differentiation_Models" / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CANDIDATES = [
    OUTPUT_DIR / "GSE226365_mat.csv",
    REPO_ROOT / "Example" / DATASET_NAME / "input" / "GSE226365_mat.csv",
    REPO_ROOT / "Example" / DATASET_NAME / "input" / "GSE226365_BioLRAF_scores.csv",
]
INPUT_FILE = next((path for path in INPUT_CANDIDATES if path.exists()), None)
if INPUT_FILE is None:
    checked = "\n".join(f"  - {path}" for path in INPUT_CANDIDATES)
    raise FileNotFoundError("GSE226365 score table was not found. Checked:\n" + checked)

CELLTYPE_ORDER = [
    "Preadipocytes",
    "Differentiating Cells",
    "Adipocytes",
]
CELLTYPE_COLORS = {
    "Preadipocytes": "#8bc96d",
    "Differentiating Cells": "#f2bc57",
    "Adipocytes": "#b79973",
}

# The uploaded result browser uses four fixed views.
CAMERAS = CAMERA_LIST[:4]

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_DIR}")

## 3. Read scores and calculate BioLRAF metrics

The normalized lineage activities are `Os_n`, `Ch_n`, and `Ad_n`. `Diff_score = 1 - MSC_ratio`. A cell is classified as `Non-dominant` when all three normalized lineage activities are within 0.10 of 1/3; otherwise, the largest activity defines the dominant lineage.

In [ ]:
scores = read_score_table(INPUT_FILE)
scores = calculate_biolraf_metrics(scores, center_delta=CENTER_DELTA)
scores["celltype"] = scores["celltype"].astype(str)

observed_celltypes = set(scores["celltype"].unique())
selected_celltypes = [ct for ct in CELLTYPE_ORDER if ct in observed_celltypes]
if not selected_celltypes:
    raise ValueError(
        "None of the expected GSE226365 cell types were found. "
        f"Observed values: {sorted(observed_celltypes)}"
    )

unexpected_celltypes = sorted(observed_celltypes.difference(CELLTYPE_ORDER))
if unexpected_celltypes:
    print(f"Excluded unexpected cell types: {unexpected_celltypes}")

scores = scores[scores["celltype"].isin(selected_celltypes)].copy()
scores["celltype"] = pd.Categorical(
    scores["celltype"], categories=selected_celltypes, ordered=True
)

display(scores.head())
display(scores["celltype"].value_counts().reindex(selected_celltypes))
display(pd.crosstab(scores["celltype"], scores["lineage_dominant"]))

## 4. Define the HTML layout and export function

Interactive HTML files retain the legend, hover information, and a fixed `Diff_score` scale. Static PNG files contain only the main 3D plot and are exported from four predefined camera positions.

In [ ]:
def add_fixed_diff_score_axis_layout(
    fig, sample_name, color_description, axis_x=0.90, axis_y0=0.25,
    axis_y1=0.73, title_x=0.43, title_y=0.965, subtitle_gap=0.045,
    font_size=12, title_size=18,
):
    tick_vals = [0.00, 0.20, 0.40, 0.60, 0.80, 1.00]
    shapes = list(fig.layout.shapes) if fig.layout.shapes else []
    annotations = list(fig.layout.annotations) if fig.layout.annotations else []

    shapes.append(dict(
        type="line", xref="paper", yref="paper",
        x0=axis_x, x1=axis_x, y0=axis_y0, y1=axis_y1,
        line=dict(color="black", width=2.2),
    ))
    for value in tick_vals:
        y = axis_y0 + (axis_y1 - axis_y0) * value
        shapes.append(dict(
            type="line", xref="paper", yref="paper",
            x0=axis_x - 0.018, x1=axis_x + 0.018, y0=y, y1=y,
            line=dict(color="black", width=1.8),
        ))
        annotations.append(dict(
            xref="paper", yref="paper", x=axis_x + 0.035, y=y,
            text=f"{value:.2f}", showarrow=False,
            xanchor="left", yanchor="middle",
            font=dict(size=font_size, color="black"),
        ))

    annotations.extend([
        dict(
            xref="paper", yref="paper", x=axis_x, y=axis_y1 + 0.055,
            text="<b>Diff_score</b>", showarrow=False,
            xanchor="center", yanchor="bottom",
            font=dict(size=font_size + 1, color="black"),
        ),
        dict(
            xref="paper", yref="paper", x=title_x, y=title_y,
            text=f"<b>{sample_name}</b>", showarrow=False,
            xanchor="center", yanchor="top",
            font=dict(size=title_size, color="black"),
        ),
        dict(
            xref="paper", yref="paper", x=title_x,
            y=title_y - subtitle_gap,
            text=f"Z-axis height indicates Diff_score; point color indicates {color_description}",
            showarrow=False, xanchor="center", yanchor="top",
            font=dict(size=font_size - 1, color="black"),
        ),
    ])
    fig.update_layout(shapes=shapes, annotations=annotations)
    return fig


def export_biolraf_plot(df, color_by, color_map, file_label, legend_title):
    plot_df = df.sort_values(color_by).copy()
    fig = plot_3d_ternary(
        df=plot_df, color_by=color_by, color_type="discrete",
        color_map=color_map, s0=S0, area_linear=True, point_size=3,
        opacity=0.5, legend_marker_size=10, title="", show=False,
    )

    fig_html = copy.deepcopy(fig)
    hidden_axis = dict(
        showbackground=False, showgrid=False, zeroline=False,
        showline=False, showticklabels=False, ticks="", title="",
    )
    fig_html.update_layout(
        width=HTML_WIDTH, height=HTML_HEIGHT, autosize=False,
        showlegend=True, margin=dict(l=0, r=95, t=95, b=25), title=None,
        scene=dict(
            domain=dict(x=[0.00, 0.82], y=[0.02, 0.86]),
            bgcolor="rgba(0,0,0,0)",
            xaxis=hidden_axis, yaxis=hidden_axis, zaxis=hidden_axis,
        ),
        legend=dict(
            title=dict(text=f"<b>{legend_title}</b>"),
            x=0.01, y=0.88, xanchor="left", yanchor="top",
            bgcolor="rgba(255,255,255,0.60)", font=dict(size=11),
        ),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    )
    fig_html = add_fixed_diff_score_axis_layout(
        fig_html, DATASET_NAME, color_description=color_by
    )

    html_file = OUTPUT_DIR / (
        f"{DATASET_NAME}_3D_ternary_{file_label}_with_fixed_Diff_score_axis.html"
    )
    fig_html.write_html(html_file, include_plotlyjs="cdn", full_html=True)
    print(f"Saved HTML: {html_file.name}")

    fig_main = configure_static_layout(fig)
    fig_main.update_layout(annotations=[], shapes=[])
    for view_number, camera in enumerate(CAMERAS, start=1):
        fig_main.update_layout(scene_camera=camera)
        png_file = OUTPUT_DIR / (
            f"{DATASET_NAME}_3D_ternary_{file_label}_{view_number}.png"
        )
        fig_main.write_image(png_file, width=300, height=300, scale=2)
        print(f"Saved PNG:  {png_file.name}")

## 5. Export the cell-type-colored results

In [ ]:
celltype_scores = scores.sort_values("celltype").copy()
celltype_color_map = {ct: CELLTYPE_COLORS[ct] for ct in selected_celltypes}

export_biolraf_plot(
    df=celltype_scores, color_by="celltype",
    color_map=celltype_color_map, file_label="celltype",
    legend_title="Celltype",
)

## 6. Export the dominant-lineage-colored results

The file label is standardized as `Lineage_dominant`. The previous uploaded filename `Lineage_dominantd_2.png` contains an extra `d` and should be renamed to `GSE226365_3D_ternary_Lineage_dominant_2.png`.

In [ ]:
export_biolraf_plot(
    df=scores, color_by="lineage_dominant",
    color_map=LINEAGE_COLORS, file_label="Lineage_dominant",
    legend_title="Lineage dominant",
)

## 7. Save the processed scores and verify output names

In [ ]:
processed_file = OUTPUT_DIR / f"{DATASET_NAME}_BioLRAF_scores_with_states.csv"
scores.to_csv(processed_file, index=False)
print(f"Saved table: {processed_file.name}")

expected_results = [
    f"{DATASET_NAME}_3D_ternary_celltype_with_fixed_Diff_score_axis.html",
    *[f"{DATASET_NAME}_3D_ternary_celltype_{i}.png" for i in range(1, 5)],
    f"{DATASET_NAME}_3D_ternary_Lineage_dominant_with_fixed_Diff_score_axis.html",
    *[f"{DATASET_NAME}_3D_ternary_Lineage_dominant_{i}.png" for i in range(1, 5)],
    processed_file.name,
]
missing_results = [name for name in expected_results if not (OUTPUT_DIR / name).exists()]
if missing_results:
    raise RuntimeError(f"Missing expected output files: {missing_results}")

print("All expected GSE226365 outputs were generated:")
for name in expected_results:
    print(f"  - {name}")